# Notebook 02 — RLM Experiments

This notebook is organised as a presentation-friendly walkthrough of Recursive
Language Models (RLMs), starting with a simple code-execution example and then
moving into recursive decomposition.

| Section | Concept demonstrated |
|---|---|
| Intro-A | **Letter counting via Python** — let the agent write code instead of guessing |
| Intro-B | **From tool use to recursion** — why code execution motivates RLMs |
| 2-A | **Needle-in-a-Haystack** — find a hidden value in a long text |
| 2-B | **Hierarchical summarisation** — summarise multiple articles |
| 2-C | **Max depth & base case** — observe the fallback to plain LLM |
| 2-D | **Prompt tracing** — inspect the exact messages sent to the LLM server |
| 2-E | **Saving metadata** — persist call-tree and prompt-trace payloads |

## Intro-A: Why start with letter counting?

A good opening example is a task that humans solve reliably, but language models
often solve inconsistently when they rely on next-token prediction alone.

Use this framing in the session:

1. Ask the audience to count a specific letter in a word such as `strawberry`.
2. Point out that a plain LLM may guess or pattern-match.
3. Then show that once the model is allowed to write Python, the task becomes deterministic.

That is the bridge to RLMs: instead of forcing the model to do everything in one
shot inside a token stream, we let it externalise reasoning into code and later
into recursive subcalls.

## Setup

In [ ]:
import os
import sys
import json
import random
import textwrap

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://host-gateway:1337/v1')
LLAMA_MODEL    = os.environ.get('LLAMA_MODEL',    'local-model')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed')

from rlm_smolagent import RLMAgent

def make_agent(max_depth=2, max_steps=10, verbose=False, capture_prompt_traces=False):
    return RLMAgent(
        base_url=LLAMA_BASE_URL,
        model_name=LLAMA_MODEL,
        api_key=OPENAI_API_KEY,
        max_depth=max_depth,
        max_steps=max_steps,
        verbose=verbose,
        capture_prompt_traces=capture_prompt_traces,
    )

print('Setup complete.')

## Helper: pretty-print the call tree

In [ ]:
def print_tree(node: dict, indent: int = 0) -> None:
    """Recursively print a call-tree dictionary produced by RLMAgent."""
    prefix = '  ' * indent
    depth  = node.get('depth', '?')
    dur    = node.get('duration_s', '?')
    prompt = node.get('prompt_preview', '')
    resp   = node.get('response_preview', '')
    print(f"{prefix}[depth={depth}] ({dur}s)")
    print(f"{prefix}  prompt  : {prompt}")
    print(f"{prefix}  response: {resp}")
    for child in node.get('children', []):
        print_tree(child, indent + 1)

## Helper: inspect prompt traces

`capture_prompt_traces=True` records the exact message payloads sent to the
OpenAI-compatible server for each CodeAgent step and each plain fallback call.
These helpers make the captured data easier to inspect in the notebook.

In [ ]:
def count_llm_requests(node: dict) -> int:
    total = len(node.get('llm_requests', []))
    for child in node.get('children', []):
        total += count_llm_requests(child)
    return total

def print_prompt_trace(result) -> None:
    print(result.format_prompt_trace())

def print_request_summary(result) -> None:
    requests = result.iter_llm_requests()
    print(f'Total outbound LLM requests: {len(requests)}')
    for index, request in enumerate(requests, start=1):
        tools = ', '.join(request.get('tool_names', [])) or 'none'
        print(
            f"{index:02d}. depth={request.get('depth')} "
            f"phase={request.get('phase')} tools={tools}"
        )

---
## Intro-B: Opening demo — let the agent write Python for counting

In this demo, we intentionally use a task that many LLMs answer unreliably when
they guess from tokens alone: counting letters inside a word.

The goal is not recursion yet. The goal is to show a more general principle:
when the model can use a Python REPL, it can convert a brittle cognitive task
into an exact computation.

In [ ]:
count_word = 'strawberry'
target_letter = 'r'

count_agent = make_agent(
    max_depth=1,
    max_steps=8,
    verbose=False,
    capture_prompt_traces=True,
 )

count_prompt = textwrap.dedent(f"""\
    Count how many times the letter '{target_letter}' appears in the word '{count_word}'.

    Important: do not answer from intuition.
    Use Python code in the REPL to compute the answer exactly, then return:
    - the final count
    - a one-sentence explanation of why using code is more reliable here

    Keep the response short.
""")

count_result = count_agent.completion(count_prompt)

print('=== Counting Demo Answer ===')
print(count_result.response)

print('\n=== Prompt Summary ===')
print_request_summary(count_result)

In [ ]:
print('\n=== Prompt Trace ===')
print_prompt_trace(count_result)

print('\n=== Raw First Request ===')
print(json.dumps(count_result.iter_llm_requests()[0], indent=2))

## Intro-C: From code execution to RLM

The counting demo shows the first step: the model can offload an exact subtask
to Python instead of approximating it in natural language.

RLMs extend this idea one level further:

1. The model can decide that a task is too large or too complex for one pass.
2. It can create smaller subproblems with `rlm_call(sub_prompt)`.
3. Each subproblem is solved by another agent instance.
4. The parent agent aggregates the child results into a final answer.

So the core idea is not only tool use. It is tool use plus self-decomposition.

---
## Experiment 2-A: Needle-in-a-Haystack

We hide a secret number inside a long passage of lorem-ipsum-like text and ask
the RLM to find it.  The RLM should split the haystack into chunks, search each
one recursively, and report the needle.

> This is the canonical "Hello World" example from the RLM paper.

In [ ]:
random.seed(7)

WORDS = (
    "the quick brown fox jumps over the lazy dog "
    "lorem ipsum dolor sit amet consectetur adipiscing elit "
).split()

def make_haystack(n_words: int = 800, needle_word: str = "SECRET_42") -> str:
    words = [random.choice(WORDS) for _ in range(n_words)]
    insert_pos = random.randint(n_words // 4, 3 * n_words // 4)
    words.insert(insert_pos, needle_word)
    return ' '.join(words)

needle  = "SECRET_42"
haystack = make_haystack(n_words=800, needle_word=needle)

print(f'Haystack length : {len(haystack.split())} words')
print(f'Needle          : "{needle}"')

In [ ]:
agent = make_agent(max_depth=2, max_steps=12, verbose=True)

prompt = textwrap.dedent(f"""\
    The text below contains the word \"{needle}\" exactly once.
    Your task: find that word and tell me the 5 words that appear immediately before it.

    Strategy:
      1. Split the text into two halves.
      2. Use rlm_call() to search each half.
      3. Return the context from whichever half found it.

    TEXT:
    {haystack}
""")

result = agent.completion(prompt)

print('\n=== RLM Answer ===')
print(result.response)

In [ ]:
print('\n=== Call Tree ===')
print_tree(result.metadata['call_tree'])

---
## Experiment 2-B: Hierarchical Summarisation

We generate three short "articles" and ask the RLM to produce:
1. A summary of each article (via recursive sub-calls).
2. A final meta-summary combining all three.

In [ ]:
ARTICLES = [
    (
        "Article A: Climate Change",
        "Global temperatures have risen by approximately 1.1°C above pre-industrial levels. "
        "Scientists warn that without drastic emission cuts, warming could exceed 2°C by 2100, "
        "leading to more frequent extreme weather events, rising sea levels, and biodiversity loss. "
        "Renewable energy adoption and policy changes are cited as the most critical levers."
    ),
    (
        "Article B: Artificial Intelligence",
        "Large language models (LLMs) have transformed natural language processing. "
        "Models with hundreds of billions of parameters can now write code, compose essays, "
        "and solve mathematical problems. Concerns around hallucination, bias, and energy "
        "consumption remain active research areas. Recursive inference techniques are emerging "
        "as a way to extend LLM capabilities beyond their context window."
    ),
    (
        "Article C: Space Exploration",
        "NASA's Artemis programme aims to return humans to the Moon by the late 2020s. "
        "SpaceX's Starship, the largest rocket ever built, is central to lunar and eventual "
        "Mars missions. Private companies have begun to lower the cost of orbital access "
        "significantly, opening new possibilities for satellite deployment and space tourism."
    ),
]

articles_text = '\n\n'.join(f'=== {title} ===\n{body}' for title, body in ARTICLES)
print(articles_text)

In [ ]:
agent = make_agent(max_depth=2, max_steps=12, verbose=True)

prompt = textwrap.dedent(f"""\
    You have three articles below.

    Step 1: Use rlm_call() three times — once per article — to get a
            one-sentence summary of each.
    Step 2: Combine the three summaries into a single 2-3 sentence
            meta-summary.

    Return only the final meta-summary.

    {articles_text}
""")

result = agent.completion(prompt)

print('\n=== Meta-Summary ===')
print(result.response)

In [ ]:
print('\n=== Call Tree ===')
print_tree(result.metadata['call_tree'])

---
## Experiment 2-C: Max Depth & Base Case

Set `max_depth=0` to force the agent to use the plain LLM fallback immediately.
This is useful for comparing the RLM answer vs. the naive single-shot answer.

In [ ]:
task = "Explain what a Recursive Language Model is in one paragraph."

# Plain single-shot (max_depth=0 → no recursion)
plain_agent = make_agent(max_depth=0, max_steps=5, verbose=False)
plain_result = plain_agent.completion(task)

# Full RLM (max_depth=2)
rlm_agent = make_agent(max_depth=2, max_steps=10, verbose=False)
rlm_result = rlm_agent.completion(task)

print('--- Plain (depth=0) ---')
print(plain_result.response)
print()
print('--- RLM (depth=2) ---')
print(rlm_result.response)

---
## Experiment 2-D: Prompt tracing and sub-agent instructions

This experiment turns on `capture_prompt_traces` so you can inspect the exact
messages sent to the LLM server at each recursive step.

What to look for:

1. The root agent prompt includes the system hint that teaches the model how to use `rlm_call()`.
2. Child calls create additional outbound requests, recorded with deeper recursion levels.
3. If a leaf call falls back to plain completion, that payload is captured too.

In [ ]:
trace_agent = make_agent(
    max_depth=2,
    max_steps=10,
    verbose=False,
    capture_prompt_traces=True,
 )

trace_prompt = textwrap.dedent("""\
    Explain how recursive summarisation works.

    Use rlm_call() exactly twice:
      1. First, ask one child call to explain what recursion contributes.
      2. Then, ask a second child call to explain why aggregation is needed.

    Combine the two child answers into a short final explanation.
""")

trace_result = trace_agent.completion(trace_prompt)

print('=== Final Answer ===')
print(trace_result.response)

print('\n=== Request Summary ===')
print_request_summary(trace_result)

print('\n=== Prompt Trace ===')
print_prompt_trace(trace_result)

In [ ]:
print('\n=== Call Tree ===')
print_tree(trace_result.metadata['call_tree'])

print('\n=== Raw Trace Payload (first request) ===')
first_request = trace_result.iter_llm_requests()[0]
print(json.dumps(first_request, indent=2))

print('\n=== Total Requests In Tree ===')
print(count_llm_requests(trace_result.metadata['call_tree']))

---
## Experiment 2-E: Saving and loading metadata

The `metadata` dict can be serialised to JSON for later analysis or
visualisation. When prompt tracing is enabled, the saved metadata also contains
the `llm_requests` payloads for each node in the recursive tree.

In [ ]:
import pathlib

log_dir = pathlib.Path('/workspace/logs')
log_dir.mkdir(parents=True, exist_ok=True)

log_file = log_dir / 'experiment_2d_prompt_trace.json'
log_file.write_text(json.dumps(trace_result.metadata, indent=2))

print(f'Metadata saved to {log_file}')

# Load and display
loaded = json.loads(log_file.read_text())
print_tree(loaded['call_tree'])
print()
print(f"Total saved LLM requests: {count_llm_requests(loaded['call_tree'])}")